<a href="https://colab.research.google.com/github/PurnaPriya11/Pyspark/blob/main/PySpark_Mini_ETL_PurnaPriya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark Mini ETL
## E-Commerce Orders Dataset


**Dataset:** `ecommerce_orders_raw.csv` · **Records:** 800 · **Core questions:** 5

### Instructions
- Run the setup cell first.
- Upload `ecommerce_orders_raw.csv` to the Colab session or place it beside this notebook.
- Keep `df_raw` unchanged.
- Create a separate DataFrame for each stage.
- Write your answers only in the code cells provided.
- This question paper includes questions and explanations, not solutions.

## Setup — Load the dataset

This cell imports the Spark functions needed for the assignment and loads the raw CSV into `df_raw`.

In [65]:
# Setup

In [66]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

sparkSession = SparkSession.builder.appName("ECommerceETL").master("local[*]").getOrCreate()
df_raw = sparkSession.read.csv(
    "ecommerce_orders_raw.csv",
    header=True, inferSchema=True, nullValue="NULL"

)

print("Source record count:", df_raw.count())


df_raw.printSchema()

Source record count: 800
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- source_system: string (nullable = true)



## Stage A — Source inspection

### Question A: Inspect the raw data

**Explanation:** Before transforming source data, inspect sample records, schema, data types, null values, and inconsistent category values. This helps identify columns that need cleaning or type conversion.

Complete the following:
1. Display the first 10 rows.
2. Print the schema and record count.
3. Identify date and numeric columns from the schema.
4. Inspect null counts across all columns.
5. Inspect distinct values in `order_status` and `country`.

**Write your code in the cell below.**

In [67]:
# Write your code for Stage A below.
df_raw.show(10)

+--------+-----------+----------------+--------------------+-------------------+-----------+--------+----------+------------+----------+-------------+------------+--------------+--------------+----------+-------+-------------+-------------+
|order_id|customer_id|   customer_name|      customer_email|       product_name|   category|quantity|unit_price|discount_pct|order_date|delivery_date|order_status|payment_method|payment_status|      city|country|sales_channel|source_system|
+--------+-----------+----------------+--------------------+-------------------+-----------+--------+----------+------------+----------+-------------+------------+--------------+--------------+----------+-------+-------------+-------------+
|ORD10001|      C1024|     Divya Kumar|divya.kumar1@exam...|      Notebook Pack| Stationery|     1.0|     299.0|           0|2025-10-09|         NULL|   Cancelled|   Credit Card|          paid|     Delhi|     IN|      Website|  marketplace|
|ORD10002|      C1246|       Vijay D

In [68]:
df_raw.printSchema()
print("Source record count:", df_raw.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- source_system: string (nullable = true)

Source record count: 800


In [69]:
date_columns = [
    field.name
    for field in df_raw.schema.fields
    if field.dataType.simpleString() == "date"
]

numeric_columns = [
    field.name
    for field in df_raw.schema.fields
    if field.dataType.simpleString() in [
        "integer", "int", "double", "float",
        "long", "bigint", "decimal", "short"
    ]
]

print("Date columns:", date_columns)
print("Numeric columns:", numeric_columns)

Date columns: ['order_date', 'delivery_date']
Numeric columns: ['quantity', 'unit_price', 'discount_pct']


In [86]:
null_counts = df_raw.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_raw.columns
])

print("Null counts:")
null_counts.show()

Null counts:
+--------+-----------+-------------+--------------+------------+--------+--------+----------+------------+----------+-------------+------------+--------------+--------------+----+-------+-------------+-------------+
|order_id|customer_id|customer_name|customer_email|product_name|category|quantity|unit_price|discount_pct|order_date|delivery_date|order_status|payment_method|payment_status|city|country|sales_channel|source_system|
+--------+-----------+-------------+--------------+------------+--------+--------+----------+------------+----------+-------------+------------+--------------+--------------+----+-------+-------------+-------------+
|       0|          0|            0|            15|           0|       0|      13|         9|           0|         0|          411|          13|             0|             0|  19|      0|            0|            0|
+--------+-----------+-------------+--------------+------------+--------+--------+----------+------------+----------+------

In [71]:
print("Distinct order_status values:")
df_raw.select("order_status").distinct().show()

print("Distinct country values:")
df_raw.select("country").distinct().show()


Distinct order_status values:
+------------+
|order_status|
+------------+
|    Returned|
|     Shipped|
|  Processing|
|   Cancelled|
|  In Transit|
|   DELIVERED|
|   Delivered|
|    Inactive|
|   delivered|
|     Pending|
|        NULL|
+------------+

Distinct country values:
+-------+
|country|
+-------+
|  india|
|  India|
|     IN|
+-------+




## Question 1 — Rename columns and add metadata

**Explanation:** ETL pipelines standardize column names and add metadata identifying a record's source. Use `withColumnRenamed()` for renaming and `withColumn()` with `lit()` for constant values.

Create `df_stage1` from `df_raw`:
1. Rename `customer_name` to `customer_full_name`.
2. Rename `order_status` to `status`.
3. Add `country_standard` with the constant value `"India"`.
4. Add `etl_source` with the constant value `"ecommerce_orders_raw.csv"`.
5. Display `order_id`, `customer_full_name`, `status`, `country_standard`, and `etl_source`.

**Write your code in the cell below.**

In [72]:
# Write your solution for Question 1 below.
df_stage1 = (
    df_raw
    .withColumnRenamed("customer_name", "customer_full_name")
    .withColumnRenamed("order_status", "status")
    .withColumn("country_standard", lit("India"))
    .withColumn("etl_source", lit("ecommerce_orders_raw.csv"))
)


df_stage1.select("order_id", "customer_full_name", "status", "country_standard", "etl_source").show(10)

+--------+------------------+----------+----------------+--------------------+
|order_id|customer_full_name|    status|country_standard|          etl_source|
+--------+------------------+----------+----------------+--------------------+
|ORD10001|       Divya Kumar| Cancelled|           India|ecommerce_orders_...|
|ORD10002|         Vijay Das|In Transit|           India|ecommerce_orders_...|
|ORD10003|  Keerthi Krishnan|In Transit|           India|ecommerce_orders_...|
|ORD10004|           Sneha R|In Transit|           India|ecommerce_orders_...|
|ORD10005|    Lavanya Sharma| Delivered|           India|ecommerce_orders_...|
|ORD10006|       Lavanya Das| DELIVERED|           India|ecommerce_orders_...|
|ORD10007|   Ramesh Krishnan|  Returned|           India|ecommerce_orders_...|
|ORD10008|          Ajay Das|   Shipped|           India|ecommerce_orders_...|
|ORD10009|    Sneha Krishnan|In Transit|           India|ecommerce_orders_...|
|ORD10010|        Nila Reddy|In Transit|           I

## Question 2 — Standardize matching values

**Explanation:** Source systems may represent the same status or country using different capitalization or abbreviations. Use `replace()` to standardize matching values and `distinct()` to inspect the cleaned categories.

Create `df_stage2` from `df_stage1`:
1. Replace `"delivered"` and `"DELIVERED"` with `"Delivered"` in `status`.
2. Replace `"india"` and `"IN"` with `"India"` in `country`.
3. Replace `"Inactive"` with `"Cancelled"` in `status`.
4. Display distinct values in `status` and `country`.

Preserve null status values as null.

**Write your code in the cell below.**

In [73]:
# Write your solution for Question 2 below.
df_stage2 = (
    df_stage1
    .replace(["delivered", "DELIVERED"], "Delivered", subset=["status"])
    .replace(["india", "IN"], "India", subset=["country"])
    .replace("Inactive", "Cancelled", subset=["status"])
)

print("Distinct status values:")
df_stage2.select("status").distinct().show()

print("Distinct country values:")
df_stage2.select("country").distinct().show()

Distinct status values:
+----------+
|    status|
+----------+
|  Returned|
|   Shipped|
|Processing|
| Cancelled|
|In Transit|
| Delivered|
|   Pending|
|      NULL|
+----------+

Distinct country values:
+-------+
|country|
+-------+
|  India|
+-------+



## Question 3 — Cast types and calculate revenue

**Explanation:** Data types must be suitable for calculations and date operations. Create derived revenue columns using arithmetic expressions, and use `alias()` to give an expression a readable output name.

Create `df_stage3` from `df_stage2`:
1. Cast `quantity` to integer.
2. Cast `unit_price` to `double` or `decimal`.
3. Cast `discount_pct` to integer.
4. Convert `order_date` and `delivery_date` to Spark `date` types using `to_date()`.
5. Create `gross_amount` = `quantity * unit_price`.
6. Create `discount_amount` = `gross_amount * discount_pct / 100`.
7. Create `net_amount` = `gross_amount - discount_amount`.
8. Use `alias()` in a `select()` expression to display `net_amount` as `order_revenue`.

Inspect the schema and sample rows.

**Write your code in the cell below.**

In [74]:
df_stage3 = (
    df_stage2
    .withColumn("quantity", col("quantity").cast("integer"))
    .withColumn("unit_price", col("unit_price").cast("decimal(10,2)"))
    .withColumn("discount_pct", col("discount_pct").cast("integer"))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("delivery_date", to_date(col("delivery_date")))
    .withColumn("gross_amount", col("quantity") * col("unit_price"))
    .withColumn("discount_amount", (col("gross_amount") * col("discount_pct")) / 100)
    .withColumn("net_amount", col("gross_amount") - col("discount_amount"))
)



In [75]:

df_stage3.select(
        "order_id",
        "quantity",
        "unit_price",
        "discount_pct",
        col("net_amount").alias("order_revenue")
)
df_stage3.printSchema()
df_stage3.show()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_full_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- country_standard: string (nullable = false)
 |-- etl_source: string (nullable = false)
 |-- gross_amount: decimal(21,2) (nullable = true)
 |-- discount_amount: decimal(36,6) (nullable = true)
 |-- net_amount: decim

In [76]:
# Write your solution for Question 3 below.


## Question 4 — Apply business rules and date logic

**Explanation:** ETL pipelines add business-friendly categories and date attributes so reporting teams can analyze order values and delivery performance.

Create `df_stage4` from `df_stage3`:
1. Create `order_value_band` using `when()` / `otherwise()`:
   - `net_amount` below ₹1,000 → `"Low"`
   - ₹1,000 to below ₹5,000 → `"Medium"`
   - ₹5,000 and above → `"High"`
2. Create `delivery_days` using `datediff(delivery_date, order_date)`.
3. Create `delivery_status`:
   - `status` is `"Delivered"` and `delivery_date` is not null → `"Delivered - Date Available"`
   - `status` is `"Delivered"` and `delivery_date` is null → `"Delivered - Date Missing"`
   - Otherwise → `"Not Delivered"`
4. Create `order_year` and `order_month` from `order_date` using `year()` and `month()`.

Display relevant columns to validate your transformations.

**Write your code in the cell below.**

In [77]:

df_stage4 = (
    df_stage3
    .withColumn(
        "order_value_band",
        when(col("net_amount") < 1000, "Low")
        .when(col("net_amount") < 5000, "Medium")
        .otherwise("High")
    )
    .withColumn(
        "delivery_days",
        datediff(col("delivery_date"), col("order_date"))
    )
    .withColumn(
        "delivery_status",
        when(
            (col("status") == "Delivered") & col("delivery_date").isNotNull(),
            "Delivered - Date Available"
        )
        .when(
            (col("status") == "Delivered") & col("delivery_date").isNull(),
            "Delivered - Date Missing"
        )
        .otherwise("Not Delivered")
    )
    .withColumn("order_year", year(col("order_date")))
    .withColumn("order_month", month(col("order_date")))
)

df_stage4.select(
    "order_id",
    "status",
    "net_amount",
    "order_value_band",
    "order_date",
    "delivery_date",
    "delivery_days",
    "delivery_status",
    "order_year",
    "order_month"
).show()

+--------+----------+------------+----------------+----------+-------------+-------------+--------------------+----------+-----------+
|order_id|    status|  net_amount|order_value_band|order_date|delivery_date|delivery_days|     delivery_status|order_year|order_month|
+--------+----------+------------+----------------+----------+-------------+-------------+--------------------+----------+-----------+
|ORD10001| Cancelled|  299.000000|             Low|2025-10-09|         NULL|         NULL|       Not Delivered|      2025|         10|
|ORD10002|In Transit|  474.050000|             Low|2025-08-14|   2025-08-24|           10|       Not Delivered|      2025|          8|
|ORD10003|In Transit|29996.000000|            High|2025-02-14|   2025-02-23|            9|       Not Delivered|      2025|          2|
|ORD10004|In Transit|  848.300000|             Low|2025-10-12|   2025-10-23|           11|       Not Delivered|      2025|         10|
|ORD10005| Delivered| 5937.300000|            High|2025

## Question 5 — Build the final reporting dataset

**Explanation:** The final ETL stage removes technical fields, filters invalid records, selects reporting columns, and sorts records for downstream analysis.

Create `df_final` from `df_stage4`:
1. Remove `source_system` and `etl_source` using `drop()`.
2. Keep records where `order_id` and `net_amount` are not null.
3. Keep records where `country` equals `"India"`.
4. Select: `order_id`, `customer_id`, `customer_full_name`, `product_name`, `category`, `quantity`, `unit_price`, `discount_pct`, `order_date`, `delivery_date`, `status`, `payment_status`, `city`, `country`, `gross_amount`, `discount_amount`, `net_amount`, `order_value_band`, `delivery_days`, `delivery_status`, `order_year`, `order_month`.
5. Sort by `order_date` descending and then `net_amount` descending.
6. Display 20 rows, print the final schema, and count the final records.

**Write your code in the cell below.**

In [78]:
# Write your solution for Question 5 below.
df_final = (
    df_stage4
    .drop("source_system", "etl_source")
    .filter(col("order_id").isNotNull() & col("net_amount").isNotNull())
    .filter(col("country") == "India")
    .select(
        "order_id",
        "customer_id",
        "customer_full_name",
        "product_name",
        "category",
        "quantity",
        "unit_price",
        "discount_pct",
        "order_date",
        "delivery_date",
        "status",
        "payment_status",
        "city",
        "country",
        "gross_amount",
        "discount_amount",
        "net_amount",
        "order_value_band",
        "delivery_days",
        "delivery_status",
        "order_year",
        "order_month"
    )
    .orderBy(col("order_date").desc(), col("net_amount").desc())
)

df_final.show(20)
df_final.printSchema()
print("Final record count:", df_final.count())


+--------+-----------+------------------+-------------------+-----------+--------+----------+------------+----------+-------------+----------+--------------+----------+-------+------------+---------------+------------+----------------+-------------+--------------------+----------+-----------+
|order_id|customer_id|customer_full_name|       product_name|   category|quantity|unit_price|discount_pct|order_date|delivery_date|    status|payment_status|      city|country|gross_amount|discount_amount|  net_amount|order_value_band|delivery_days|     delivery_status|order_year|order_month|
+--------+-----------+------------------+-------------------+-----------+--------+----------+------------+----------+-------------+----------+--------------+----------+-------+------------+---------------+------------+----------------+-------------+--------------------+----------+-----------+
|ORD10670|      C1166|       Priya Patel|       Office Chair|  Furniture|       4|   7499.00|          15|2026-10-03| 

## Final ETL Quality Checks — Extension

**Explanation:** Quality checks validate the output and produce basic reporting metrics. These are extension questions if you have covered `groupBy()` and aggregate functions.

1. How many records remain in `df_final`?
2. What is the total net revenue?
3. How many orders exist in each `status`?
4. What is the total net revenue by `category`?
5. How many orders have a missing `delivery_date`?
6. Which 10 orders have the highest `net_amount`?
7. How many orders exist for each `order_year` and `order_month`?

**Write your code in the cell below.**

In [79]:
# Write your ETL quality-check solutions below.
print("Final record count:", df_final.count())

Final record count: 778


In [80]:
df_final.agg(sum("net_amount").alias("total_net_revenue")).show()

+-----------------+
|total_net_revenue|
+-----------------+
|   7066817.050000|
+-----------------+



In [81]:
df_final.groupBy("status").count().show()

+----------+-----+
|    status|count|
+----------+-----+
|  Returned|   77|
|   Shipped|   82|
|Processing|   77|
|      NULL|   13|
| Cancelled|  158|
|In Transit|   88|
| Delivered|  201|
|   Pending|   82|
+----------+-----+



In [82]:
df_final.groupBy("category").agg(sum("net_amount").alias("total_net_revenue")).show()

+-----------+-----------------+
|   category|total_net_revenue|
+-----------+-----------------+
| Stationery|     49200.450000|
|Electronics|   2990439.700000|
|  Furniture|   2453782.300000|
|Accessories|   1206616.250000|
|  Lifestyle|    366778.350000|
+-----------+-----------------+



In [83]:
df_final.filter(col("delivery_date").isNull()).count()

397

In [84]:
df_final.orderBy(col("net_amount").desc()).show(10)

+--------+-----------+------------------+---------------+-----------+--------+----------+------------+----------+-------------+----------+--------------+----------+-------+------------+---------------+------------+----------------+-------------+--------------------+----------+-----------+
|order_id|customer_id|customer_full_name|   product_name|   category|quantity|unit_price|discount_pct|order_date|delivery_date|    status|payment_status|      city|country|gross_amount|discount_amount|  net_amount|order_value_band|delivery_days|     delivery_status|order_year|order_month|
+--------+-----------+------------------+---------------+-----------+--------+----------+------------+----------+-------------+----------+--------------+----------+-------+------------+---------------+------------+----------------+-------------+--------------------+----------+-----------+
|ORD10073|      C1015|       Ajay Sharma|Monitor 24 inch|Electronics|       5|  10999.00|           0|2025-04-27|   2025-05-04| De

In [85]:
df_final.groupBy("order_year", "order_month").count().show()

+----------+-----------+-----+
|order_year|order_month|count|
+----------+-----------+-----+
|      2025|          5|   30|
|      2025|          6|   52|
|      2026|          5|   30|
|      2025|         10|   35|
|      2026|          2|   36|
|      2025|          9|   24|
|      2025|          7|   41|
|      2026|          9|   42|
|      2026|         10|    4|
|      2026|          7|   43|
|      2025|          8|   34|
|      2026|          3|   28|
|      2025|          3|   38|
|      2026|          6|   29|
|      2025|         12|   37|
|      2025|          4|   42|
|      2026|          8|   38|
|      2025|         11|   45|
|      2025|          1|   27|
|      2026|          1|   42|
+----------+-----------+-----+
only showing top 20 rows


## Submission Checklist

- [ ] Stage A source inspection completed.
- [ ] `df_stage1` through `df_stage4` created.
- [ ] `df_final` created and validated.
- [ ] Schema and record counts checked.
- [ ] Quality-check questions attempted.
- [ ] Briefly documented the data-quality issues found.